### **1. Price vs Quantity (basic relationship)**

In [0]:
%sql
SELECT
    Description,
    AVG(Price) AS avg_price,
    AVG(Quantity) AS avg_qty
FROM retail
GROUP BY Description
HAVING avg_price < 1000 AND avg_qty < 1000

-- removing outliers shows that products with price 0 to 80 have most demand at upto 400, this is like total elastic demand, however some outliers are also visible.

Databricks visualization. Run in Databricks to view.

### **2. Correlation overall**

In [0]:
from pyspark.sql.functions import corr

df = spark.read.table("retail")

df.groupBy("Description").agg(
    corr("Price", "Quantity").alias("corr")
).show(1000)

In [0]:
df.groupBy("Description").agg(
    corr("Price", "Quantity").alias("corr")
).filter("corr > 0").orderBy("corr", ascending=False).show(100)

In [0]:
from pyspark.sql.functions import corr, when, col

corr_df = df.groupBy("Description").agg(
    corr("Price", "Quantity").alias("corr")
)

bucketed = corr_df.withColumn(
    "category",
    when(col("corr") == 1, "perfect_positive")
    .when(col("corr") > 0.5, "strong_positive")
    .when((col("corr") > 0) & (col("corr") <= 0.5), "weak_positive")
    .when(col("corr") == 0, "no_relation")
    .when((col("corr") < 0) & (col("corr") >= -0.5), "weak_negative")
    .when(col("corr") < -0.5, "strong_negative")
)

bucketed.groupBy("category").count().show()

### Correlation analysis indicates mixed price–quantity relationships across products. However, since correlation does not capture sensitivity, elasticity is computed to measure true demand response.

### **3. Elasticity Calculation**

In [0]:
%sql

WITH data AS (
    SELECT
        Description,
        DATE(InvoiceDate) AS dt,
        AVG(Price) AS price,
        SUM(Quantity) AS qty
    FROM retail
    GROUP BY Description, dt
),

calc AS (
    SELECT
        Description,
        dt,
        price,
        qty,
        LAG(price) OVER (PARTITION BY Description ORDER BY dt) AS prev_price,
        LAG(qty) OVER (PARTITION BY Description ORDER BY dt) AS prev_qty
    FROM data
),

elasticity_row AS (
    SELECT
        Description,
        try_divide(
            try_divide(qty - prev_qty, prev_qty),
            try_divide(price - prev_price, prev_price)
        ) AS elasticity
    FROM calc
    WHERE 
        prev_price IS NOT NULL
        AND prev_price != 0
        AND ABS((price - prev_price)/prev_price) > 0.1
),

filtered AS (
    SELECT *
    FROM elasticity_row
    WHERE ABS(elasticity) < 10
),

final AS (
    SELECT
        Description,
        AVG(elasticity) AS elasticity
    FROM filtered
    GROUP BY Description
)

SELECT
    Description,
    elasticity,
    CASE
        WHEN elasticity > 0 THEN 'anomalous_positive'
        WHEN ABS(elasticity) > 1 THEN 'elastic'
        WHEN ABS(elasticity) = 1 THEN 'unitary'
        WHEN ABS(elasticity) < 1 THEN 'inelastic'
    END AS category
FROM final

Databricks visualization. Run in Databricks to view.

### The following conclusions are made based on elasticity outcomes:-
1. The anomalous positive means elasticity being completely positive, that means the Giffen or veblen goods whose demand seems to be increasing when price increases.

2. The elastic demand meant any elasticity greater then 1, but since positive are filtered out in CASE 1, then only negative will be in elastic demand cases, hereby stating the law of demand, the quantity changes in opposite direction of price change.

3. The inelastic demand meant that the elasticity is between 
-1 < inelastic < +1 . Example -0.3. This means that quantitiy demanded change is very less in terms of price change. That means people are willing to still pay for that good even if price is relatively higher.

4. Total 1979 inelastic goods, 1722 elastic goods and 657 giffen goods are found in the data.

In [0]:
_sqldf.write.format("delta").mode("overwrite").saveAsTable("elasticity_table")

#saved the table as a delta

In [0]:
%sql
SELECT * FROM elasticity_table LIMIT 10

This table is further used in next notebook named ADVANCED DEMAND ANALYSIS